# LSTM Example: Sine Wave Prediction
Predicting future values of a sine wave from past values — a simple sequence-to-value regression task.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 1. Generate Data
We create a sine wave and chop it into overlapping windows of length `seq_len`. Each window is an input sequence, and the target is the next value after the window.

In [ ]:
# Generate sine wave
t = np.linspace(0, 100, 2000)
series = np.sin(t)

# Create sequences: given seq_len past values, predict the next value
seq_len = 50

def create_sequences(data, seq_len):
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        xs.append(data[i:i+seq_len])
        ys.append(data[i+seq_len])
    return np.array(xs), np.array(ys)

X, y = create_sequences(series, seq_len)

# Train/test split (80/20)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Convert to PyTorch tensors
# LSTM expects input shape: (batch, seq_len, input_dim)
X_train = torch.FloatTensor(X_train).unsqueeze(-1)  # add feature dim
y_train = torch.FloatTensor(y_train).unsqueeze(-1)
X_test = torch.FloatTensor(X_test).unsqueeze(-1)
y_test = torch.FloatTensor(y_test).unsqueeze(-1)

print(f"X_train shape: {X_train.shape}")  # (batch, seq_len, 1)
print(f"y_train shape: {y_train.shape}")  # (batch, 1)

## 2. Define the LSTM Model
The architecture is:
1. **LSTM layer** — processes the sequence, outputs a hidden state at each timestep
2. **Linear layer** — maps the final hidden state to a single predicted value

The key thing to understand: the LSTM sees the input one timestep at a time, maintaining a hidden state $h_t$ and a cell state $c_t$ that carry information forward through the sequence.

In [ ]:
class SineLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=1, output_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # LSTM: takes (batch, seq_len, input_dim) -> (batch, seq_len, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True  # so input is (batch, seq, features) not (seq, batch, features)
        )
        
        # Linear: maps final hidden state to prediction
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # Initialise hidden state and cell state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        
        # LSTM forward pass
        # out shape: (batch, seq_len, hidden_dim) — hidden state at every timestep
        # (h_n, c_n) — final hidden and cell states
        out, (h_n, c_n) = self.lstm(x, (h0, c0))
        
        # We only care about the last timestep's hidden state
        last_hidden = out[:, -1, :]  # shape: (batch, hidden_dim)
        
        # Map to prediction
        prediction = self.fc(last_hidden)  # shape: (batch, output_dim)
        return prediction

model = SineLSTM()
print(model)

## 3. Training Loop
Standard PyTorch training: forward pass → compute loss → backward pass → update weights.

In [ ]:
criterion = nn.MSELoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50
batch_size = 64

# Simple batching with a DataLoader
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

losses = []
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        # Forward pass
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Backward pass
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.show()

## 4. Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    test_preds = model(X_test)
    test_loss = criterion(test_preds, y_test)
    print(f"Test MSE: {test_loss.item():.6f}")

# Plot predictions vs ground truth
plt.figure(figsize=(12, 4))
plt.plot(y_test.numpy(), label="Ground Truth", alpha=0.7)
plt.plot(test_preds.numpy(), label="Predictions", alpha=0.7)
plt.legend()
plt.title("LSTM Sine Wave Predictions vs Ground Truth")
plt.show()